# 05 select leiden resolution

**Purpose:** Evaluate Leiden resolutions and select the r=0.6 solution.

**Original analysis notebook:** `05_clustering_resolution_QC_HVG8000_lowmem.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.

# Clustering resolution QC and resolution selection — HVG8000 scVI low-memory

هدف این نوت‌بوک این است که **به‌صورت اصولی** چند resolution مختلف Leiden را روی latent space مدل scVI بررسی کنیم و بهترین resolution را برای marker DE و annotation انتخاب کنیم.

اصل تصمیم‌گیری:
- دنبال عدد قبلی مثل 24 cluster نیستیم.
- resolution مناسب باید از نظر **marker separation، اندازه کلاسترها، batch/dataset bias، patient/sample dominance، condition composition، UMAP structure و stability/QC** قابل دفاع باشد.
- خروجی نهایی این نوت‌بوک یک جدول advisor است که candidate resolutionها را پیشنهاد می‌دهد؛ تصمیم نهایی با بررسی جدول‌ها و figureها گرفته می‌شود.

ورودی اصلی:
`GBM_core_4datasets_HVG8000_scVI_UMAP_Leiden_lowmem_Xcounts.h5ad`

ورودی full-gene فقط برای marker dotplot و module scoring استفاده می‌شود:
`GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad`

In [ ]:
# =========================
# 0) SETUP
# =========================

import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

import scanpy as sc
import anndata as ad

import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -------------------------
# Project paths
# -------------------------

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_DIR = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
METADATA_DIR = PROJECT_DIR / "metadata"
FIGURES_DIR = PROJECT_DIR / "figures"
RESULTS_DIR = PROJECT_DIR / "results"

for d in [PROCESSED_DIR, METADATA_DIR, FIGURES_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

SCVI_H5AD = PROCESSED_DIR / "integrated" / "GBM_core_4datasets_HVG8000_scVI_UMAP_Leiden_lowmem_Xcounts.h5ad"
FULLGENE_H5AD = PROCESSED_DIR / "integrated" / "GBM_core_4datasets_fullgene_postQC_merged_counts_metadata.h5ad"

OUT_H5AD = PROCESSED_DIR / "integrated" / "GBM_core_4datasets_HVG8000_scVI_clustering_resolution_QC_lowmem.h5ad"

# -------------------------
# Output TSVs
# -------------------------

SWEEP_SUMMARY_TSV = METADATA_DIR / "clustering_resolution_sweep_HVG8000_lowmem.tsv"
CLUSTER_SIZES_TSV = METADATA_DIR / "clustering_resolution_cluster_sizes_HVG8000_lowmem.tsv"
COMPOSITION_TSV = METADATA_DIR / "clustering_resolution_composition_metrics_HVG8000_lowmem.tsv"
COMPOSITION_SUMMARY_TSV = METADATA_DIR / "clustering_resolution_composition_summary_HVG8000_lowmem.tsv"
SILHOUETTE_TSV = METADATA_DIR / "clustering_resolution_silhouette_metrics_HVG8000_lowmem.tsv"
MARKER_MODULE_TSV = METADATA_DIR / "clustering_resolution_marker_module_summary_HVG8000_lowmem.tsv"
ADVISOR_TSV = METADATA_DIR / "clustering_resolution_advisor_table_HVG8000_lowmem.tsv"
RECOMMENDATION_TSV = METADATA_DIR / "clustering_resolution_recommended_resolution_HVG8000_lowmem.tsv"
MANIFEST_TSV = METADATA_DIR / "clustering_resolution_QC_output_manifest_HVG8000_lowmem.tsv"

# -------------------------
# Parameters
# -------------------------

EMBED_KEY = "X_scVI"
UMAP_KEY = "X_umap"

N_NEIGHBORS = 30
NEIGHBORS_KEY = f"neighbors_scVI_n{N_NEIGHBORS}"

# Broad sweep; we are NOT targeting a fixed number of clusters.
RESOLUTIONS = [
    0.10, 0.15, 0.20, 0.25, 0.30, 0.35,
    0.40, 0.45, 0.50, 0.55, 0.60, 0.70,
    0.80, 0.90, 1.00, 1.20
]

RANDOM_STATE = 42

# For silhouette only; full silhouette on 351k cells can be too slow.
SILHOUETTE_MAX_CELLS = 30000

# For figures: after advisor scoring, we only draw detailed dotplots/heatmaps for top candidate resolutions.
N_TOP_CANDIDATE_FIGURES = 4

sc.settings.verbosity = 2
sc.settings.figdir = str(FIGURES_DIR)

def write_tsv_replace(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    df.to_csv(path, sep="\t", index=False)
    print(f"[SAVED] {path} shape={df.shape}")

def write_h5ad_replace(adata_obj, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.exists():
        path.unlink()
    adata_obj.write_h5ad(path)
    print(f"[SAVED] {path}")

def normalized_entropy(values):
    s = pd.Series(values).astype(str)
    counts = s.value_counts()
    if counts.shape[0] <= 1:
        return 0.0
    p = counts / counts.sum()
    ent = -np.sum(p * np.log(p + 1e-12))
    return float(ent / np.log(counts.shape[0]))

def safe_fraction_top(values):
    s = pd.Series(values).astype(str)
    if len(s) == 0:
        return np.nan, "NA"
    vc = s.value_counts()
    return float(vc.iloc[0] / vc.sum()), str(vc.index[0])

print("SCVI_H5AD:", SCVI_H5AD, "exists:", SCVI_H5AD.exists())
print("FULLGENE_H5AD:", FULLGENE_H5AD, "exists:", FULLGENE_H5AD.exists())

In [ ]:
# =========================
# 1) READ scVI OBJECT AND VALIDATE INPUT
# =========================

if not SCVI_H5AD.exists():
    raise FileNotFoundError(f"SCVI h5ad not found: {SCVI_H5AD}")

adata = sc.read_h5ad(SCVI_H5AD)
print(adata)

required_obs = ["core_dataset", "condition", "patient_id", "sample_id", "scvi_batch"]
missing_obs = [c for c in required_obs if c not in adata.obs.columns]
if missing_obs:
    raise KeyError(f"Missing required obs columns: {missing_obs}")

if EMBED_KEY not in adata.obsm.keys():
    raise KeyError(f"{EMBED_KEY} not found in adata.obsm. Available: {list(adata.obsm.keys())}")

for col in required_obs:
    adata.obs[col] = adata.obs[col].astype(str).fillna("unknown")

# Validate that this is the low-memory scVI object where counts are in X
x_sample = adata.X[:2000, :2000]
vals = x_sample.data if sp.issparse(x_sample) else np.asarray(x_sample).ravel()
vals = vals[np.isfinite(vals)]

input_qc = pd.DataFrame([{
    "n_cells": adata.n_obs,
    "n_genes": adata.n_vars,
    "X_is_sparse": sp.issparse(adata.X),
    "layers": ",".join(list(adata.layers.keys())),
    "has_counts_layer": "counts" in adata.layers,
    "counts_source_expected_for_scvi": "adata.X",
    "X_sample_min": float(vals.min()) if len(vals) else np.nan,
    "X_sample_max": float(vals.max()) if len(vals) else np.nan,
    "X_sample_integer_like": bool(np.allclose(vals, np.round(vals))) if len(vals) else False,
    "X_sample_any_negative": bool(np.any(vals < 0)) if len(vals) else False,
    "latent_key": EMBED_KEY,
    "latent_shape": str(adata.obsm[EMBED_KEY].shape),
    "latent_nan_count": int(np.isnan(adata.obsm[EMBED_KEY]).sum()),
}])

INPUT_QC_TSV = METADATA_DIR / "clustering_resolution_input_qc_HVG8000_lowmem.tsv"
write_tsv_replace(input_qc, INPUT_QC_TSV)
display(input_qc)

if not bool(input_qc.loc[0, "X_sample_integer_like"]):
    print("[WARNING] adata.X does not look integer-like in the sampled block. Check whether counts are really in X.")

if bool(input_qc.loc[0, "X_sample_any_negative"]):
    raise ValueError("adata.X contains negative values, which is unexpected for count input.")

print("Input validated.")

In [ ]:
# =========================
# 2) BUILD / REBUILD NEIGHBOR GRAPH AND RUN LEIDEN RESOLUTION SWEEP
# =========================

print(f"Building neighbors using {EMBED_KEY}, n_neighbors={N_NEIGHBORS}")

sc.pp.neighbors(
    adata,
    use_rep=EMBED_KEY,
    n_neighbors=N_NEIGHBORS,
    key_added=NEIGHBORS_KEY,
    random_state=RANDOM_STATE,
)

sweep_rows = []
cluster_size_rows = []

for res in RESOLUTIONS:
    res_str = str(res).replace(".", "_")
    key = f"leiden_scvi_HVG8000_r{res_str}"

    print(f"Running Leiden resolution={res} -> {key}")
    sc.tl.leiden(
        adata,
        resolution=res,
        key_added=key,
        neighbors_key=NEIGHBORS_KEY,
        random_state=RANDOM_STATE,
    )

    counts = adata.obs[key].astype(str).value_counts().sort_index(
        key=lambda idx: [int(x) if str(x).isdigit() else str(x) for x in idx]
    )

    for cl, n in counts.items():
        cluster_size_rows.append({
            "resolution": res,
            "cluster_key": key,
            "cluster": str(cl),
            "n_cells": int(n),
            "fraction_cells": float(n / adata.n_obs),
        })

    sweep_rows.append({
        "resolution": res,
        "cluster_key": key,
        "n_clusters": int(counts.shape[0]),
        "min_cluster_size": int(counts.min()),
        "max_cluster_size": int(counts.max()),
        "median_cluster_size": float(counts.median()),
        "mean_cluster_size": float(counts.mean()),
        "n_clusters_lt_50_cells": int((counts < 50).sum()),
        "n_clusters_lt_100_cells": int((counts < 100).sum()),
        "n_clusters_lt_500_cells": int((counts < 500).sum()),
        "n_clusters_lt_1000_cells": int((counts < 1000).sum()),
        "frac_cells_in_clusters_lt_100": float(counts[counts < 100].sum() / adata.n_obs),
        "frac_cells_in_clusters_lt_500": float(counts[counts < 500].sum() / adata.n_obs),
        "frac_cells_in_clusters_lt_1000": float(counts[counts < 1000].sum() / adata.n_obs),
    })

sweep_df = pd.DataFrame(sweep_rows)
cluster_sizes_df = pd.DataFrame(cluster_size_rows)

write_tsv_replace(sweep_df, SWEEP_SUMMARY_TSV)
write_tsv_replace(cluster_sizes_df, CLUSTER_SIZES_TSV)

display(sweep_df)
display(cluster_sizes_df.head())

In [ ]:
# =========================
# 3) CLUSTER COMPOSITION / BIAS METRICS
# =========================

composition_rows = []

metadata_cols = {
    "core_dataset": "dataset",
    "condition": "condition",
    "patient_id": "patient",
    "sample_id": "sample",
}

for _, row in sweep_df.iterrows():
    res = float(row["resolution"])
    key = row["cluster_key"]

    for cl, sub_idx in adata.obs.groupby(key, observed=True).groups.items():
        obs_sub = adata.obs.loc[sub_idx]
        out = {
            "resolution": res,
            "cluster_key": key,
            "cluster": str(cl),
            "n_cells": int(obs_sub.shape[0]),
            "fraction_cells": float(obs_sub.shape[0] / adata.n_obs),
        }

        for col, label in metadata_cols.items():
            top_frac, top_label = safe_fraction_top(obs_sub[col])
            out[f"top_{label}"] = top_label
            out[f"top_{label}_fraction"] = top_frac
            out[f"{label}_normalized_entropy"] = normalized_entropy(obs_sub[col])

        composition_rows.append(out)

composition_df = pd.DataFrame(composition_rows)
write_tsv_replace(composition_df, COMPOSITION_TSV)

summary_rows = []
for (res, key), sub in composition_df.groupby(["resolution", "cluster_key"], observed=True):
    weights = sub["n_cells"].values / sub["n_cells"].sum()

    def weighted_mean(col):
        return float(np.sum(sub[col].values * weights))

    summary_rows.append({
        "resolution": res,
        "cluster_key": key,
        "n_clusters": int(sub.shape[0]),
        "mean_top_dataset_fraction_weighted": weighted_mean("top_dataset_fraction"),
        "mean_top_condition_fraction_weighted": weighted_mean("top_condition_fraction"),
        "mean_top_patient_fraction_weighted": weighted_mean("top_patient_fraction"),
        "mean_top_sample_fraction_weighted": weighted_mean("top_sample_fraction"),
        "mean_dataset_entropy_weighted": weighted_mean("dataset_normalized_entropy"),
        "mean_condition_entropy_weighted": weighted_mean("condition_normalized_entropy"),
        "mean_patient_entropy_weighted": weighted_mean("patient_normalized_entropy"),
        "mean_sample_entropy_weighted": weighted_mean("sample_normalized_entropy"),
        "n_clusters_dataset_fraction_gt_0_80": int((sub["top_dataset_fraction"] > 0.80).sum()),
        "n_clusters_condition_fraction_gt_0_80": int((sub["top_condition_fraction"] > 0.80).sum()),
        "n_clusters_patient_fraction_gt_0_80": int((sub["top_patient_fraction"] > 0.80).sum()),
        "n_clusters_sample_fraction_gt_0_80": int((sub["top_sample_fraction"] > 0.80).sum()),
        "max_top_dataset_fraction": float(sub["top_dataset_fraction"].max()),
        "max_top_condition_fraction": float(sub["top_condition_fraction"].max()),
        "max_top_patient_fraction": float(sub["top_patient_fraction"].max()),
        "max_top_sample_fraction": float(sub["top_sample_fraction"].max()),
    })

composition_summary_df = pd.DataFrame(summary_rows)
write_tsv_replace(composition_summary_df, COMPOSITION_SUMMARY_TSV)

display(composition_summary_df)

In [ ]:
# =========================
# 4) SILHOUETTE METRICS ON SUBSAMPLED scVI LATENT
# =========================

# This is an approximate diagnostic, not the final decision by itself.
# cluster silhouette: higher can mean better separation, but can favor over-separated graphs.
# dataset/condition silhouette: very high values suggest remaining dataset/condition separation in latent space.

try:
    from sklearn.metrics import silhouette_score

    rng = np.random.default_rng(RANDOM_STATE)
    n_sub = min(SILHOUETTE_MAX_CELLS, adata.n_obs)
    sub_idx = rng.choice(adata.n_obs, size=n_sub, replace=False)
    X_sub = np.asarray(adata.obsm[EMBED_KEY][sub_idx, :], dtype=np.float32)

    sil_rows = []

    # Dataset/condition silhouette on latent space
    for label_col in ["core_dataset", "condition"]:
        labels = adata.obs.iloc[sub_idx][label_col].astype(str).values
        if len(np.unique(labels)) > 1:
            sil = silhouette_score(X_sub, labels, metric="euclidean")
        else:
            sil = np.nan
        sil_rows.append({
            "resolution": "NA",
            "cluster_key": "NA",
            "metric": f"{label_col}_silhouette_on_X_scVI_subsample",
            "n_cells_subsample": int(n_sub),
            "value": float(sil) if pd.notna(sil) else np.nan,
            "interpretation": "High value may indicate separation by metadata label; lower/near-zero usually indicates better mixing for technical labels.",
        })

    # Cluster silhouette per resolution
    for _, row in sweep_df.iterrows():
        key = row["cluster_key"]
        labels = adata.obs.iloc[sub_idx][key].astype(str).values
        if len(np.unique(labels)) > 1 and len(np.unique(labels)) < len(labels):
            sil = silhouette_score(X_sub, labels, metric="euclidean")
        else:
            sil = np.nan

        sil_rows.append({
            "resolution": float(row["resolution"]),
            "cluster_key": key,
            "metric": "cluster_silhouette_on_X_scVI_subsample",
            "n_cells_subsample": int(n_sub),
            "value": float(sil) if pd.notna(sil) else np.nan,
            "interpretation": "Higher can indicate sharper clustering, but it must be interpreted with marker and bias QC.",
        })

    silhouette_df = pd.DataFrame(sil_rows)
    write_tsv_replace(silhouette_df, SILHOUETTE_TSV)
    display(silhouette_df)

except Exception as e:
    silhouette_df = pd.DataFrame([{
        "error": repr(e),
        "note": "Silhouette metrics failed or sklearn is unavailable. The rest of the notebook can continue."
    }])
    write_tsv_replace(silhouette_df, SILHOUETTE_TSV)
    print("[WARNING] Silhouette calculation skipped:", repr(e))

In [ ]:
# =========================
# 5) BASIC UMAP FIGURES FOR METADATA AND RESOLUTIONS
# =========================

# Compute UMAP if missing.
if UMAP_KEY not in adata.obsm.keys():
    print("X_umap missing. Computing UMAP from scVI neighbors.")
    sc.tl.umap(adata, neighbors_key=NEIGHBORS_KEY, random_state=RANDOM_STATE, min_dist=0.3)
else:
    print("X_umap already present.")

# UMAP colored by metadata
metadata_fig_path = FIGURES_DIR / "clustering_QC_HVG8000_UMAP_metadata_lowmem.png"
with plt.rc_context({"figure.figsize": (6, 5)}):
    sc.pl.umap(
        adata,
        color=["core_dataset", "condition"],
        wspace=0.4,
        show=False,
    )
    plt.savefig(metadata_fig_path, dpi=180, bbox_inches="tight")
    plt.close()
print("[SAVED]", metadata_fig_path)

# UMAP for all sweep resolutions in batches to avoid giant figures.
resolution_keys = list(sweep_df["cluster_key"])

batch_size = 4
for i in range(0, len(resolution_keys), batch_size):
    keys = resolution_keys[i:i+batch_size]
    fig_path = FIGURES_DIR / f"clustering_QC_HVG8000_UMAP_resolution_batch_{i//batch_size+1}_lowmem.png"
    with plt.rc_context({"figure.figsize": (5 * len(keys), 4.5)}):
        sc.pl.umap(
            adata,
            color=keys,
            wspace=0.35,
            legend_loc="on data",
            legend_fontsize=6,
            show=False,
        )
        plt.savefig(fig_path, dpi=170, bbox_inches="tight")
        plt.close()
    print("[SAVED]", fig_path)

In [ ]:
# =========================
# 6) LOAD FULL-GENE MARKER PANEL SUBSET FOR BIOLOGICAL QC
# =========================

# We use the full-gene object only for a small marker-gene subset.
# This avoids using the HVG-only matrix for marker dotplots, where some important markers may be absent.

marker_panels = {
    "Tumor_AC_like": ["GFAP", "AQP4", "ALDH1L1", "SLC1A3", "CLU", "APOE"],
    "Tumor_OPC_like": ["PDGFRA", "OLIG1", "OLIG2", "SOX10", "CSPG4", "BCAN"],
    "Tumor_NPC_like": ["SOX2", "NES", "VIM", "FABP7", "DLL3", "ASCL1"],
    "Tumor_MES_like": ["CHI3L1", "CD44", "ANXA1", "VIM", "SERPINE1", "TNC", "EMP3"],
    "Cycling": ["MKI67", "TOP2A", "PCNA", "UBE2C", "STMN1", "HMGB2"],
    "Hypoxia_stress": ["VEGFA", "CA9", "NDRG1", "DDIT4", "HILPDA", "ENO1"],
    "Myeloid": ["PTPRC", "LYZ", "C1QA", "C1QB", "C1QC", "TYROBP", "AIF1", "CSF1R"],
    "Inflammatory_myeloid": ["IL1B", "CCL3", "CCL4", "CXCL8", "NFKBIA", "TNF"],
    "T_NK": ["CD3D", "CD3E", "TRAC", "IL7R", "NKG7", "GNLY", "GZMB"],
    "B_Plasma": ["MS4A1", "CD79A", "CD79B", "MZB1", "JCHAIN", "SDC1"],
    "Oligodendrocyte": ["MBP", "PLP1", "MOG", "MAG", "MOBP", "CLDN11"],
    "Endothelial": ["PECAM1", "VWF", "KDR", "FLT1", "CLDN5", "RAMP2"],
    "Mural_Pericyte": ["PDGFRB", "RGS5", "ACTA2", "TAGLN", "MCAM", "CSPG4"],
    "Fibroblast_ECM": ["COL1A1", "COL1A2", "DCN", "LUM", "COL3A1", "FAP"],
}

all_marker_genes = sorted(set(g for genes in marker_panels.values() for g in genes))

if not FULLGENE_H5AD.exists():
    print(f"[WARNING] Full-gene h5ad not found: {FULLGENE_H5AD}")
    print("Marker module scoring and dotplots will be skipped.")
    adata_marker = None
else:
    print("Reading marker subset from full-gene object:", FULLGENE_H5AD)
    adata_full_backed = sc.read_h5ad(FULLGENE_H5AD, backed="r")

    present_genes = [g for g in all_marker_genes if g in adata_full_backed.var_names]
    missing_genes = [g for g in all_marker_genes if g not in adata_full_backed.var_names]

    marker_presence = pd.DataFrame({
        "gene": all_marker_genes,
        "present_in_fullgene": [g in present_genes for g in all_marker_genes],
    })
    MARKER_PRESENCE_TSV = METADATA_DIR / "clustering_resolution_marker_panel_presence_HVG8000_lowmem.tsv"
    write_tsv_replace(marker_presence, MARKER_PRESENCE_TSV)

    print("Marker genes present:", len(present_genes), "/", len(all_marker_genes))
    if missing_genes:
        print("Missing marker genes:", missing_genes)

    if len(present_genes) == 0:
        raise ValueError("None of the marker genes were found in the full-gene object.")

    # Subset only marker genes into memory.
    adata_marker = adata_full_backed[:, present_genes].to_memory()
    adata_full_backed.file.close()
    del adata_full_backed

    # Align cells and transfer clustering columns + metadata from scVI object.
    common = adata_marker.obs_names.intersection(adata.obs_names)
    print("Common cells marker/fullgene vs scVI:", len(common), "/", adata.n_obs)

    adata_marker = adata_marker[common].copy()
    adata_scvi_aligned = adata[common].copy()

    transfer_cols = ["core_dataset", "condition", "patient_id", "sample_id", "scvi_batch"] + resolution_keys
    for col in transfer_cols:
        if col in adata_scvi_aligned.obs.columns:
            adata_marker.obs[col] = adata_scvi_aligned.obs[col].astype(str).values

    # Use counts for marker subset.
    if "counts" in adata_marker.layers:
        adata_marker.X = adata_marker.layers["counts"].copy()

    # Normalize/log transform only the small marker subset.
    sc.pp.normalize_total(adata_marker, target_sum=1e4)
    sc.pp.log1p(adata_marker)

    print(adata_marker)
    del adata_scvi_aligned
    gc.collect()

In [ ]:
# =========================
# 7) MARKER MODULE SCORING PER RESOLUTION
# =========================

if adata_marker is None:
    marker_module_df = pd.DataFrame([{
        "note": "Marker module scoring skipped because full-gene marker subset was unavailable."
    }])
    write_tsv_replace(marker_module_df, MARKER_MODULE_TSV)

else:
    # Score marker modules using present genes only.
    present_marker_panels = {}
    for module, genes in marker_panels.items():
        present = [g for g in genes if g in adata_marker.var_names]
        if len(present) >= 2:
            present_marker_panels[module] = present

    print("Marker modules with >=2 present genes:")
    for m, genes in present_marker_panels.items():
        print(m, genes)

    for module, genes in present_marker_panels.items():
        score_col = f"score_{module}"
        sc.tl.score_genes(
            adata_marker,
            gene_list=genes,
            score_name=score_col,
            use_raw=False,
            random_state=RANDOM_STATE,
        )

    score_cols = [f"score_{m}" for m in present_marker_panels.keys()]

    module_rows = []
    for _, row in sweep_df.iterrows():
        res = float(row["resolution"])
        key = row["cluster_key"]

        cluster_means = adata_marker.obs.groupby(key, observed=True)[score_cols].mean()

        for cl, vals in cluster_means.iterrows():
            sorted_vals = vals.sort_values(ascending=False)
            top_module = sorted_vals.index[0].replace("score_", "")
            second_module = sorted_vals.index[1].replace("score_", "") if len(sorted_vals) > 1 else "NA"
            top_score = float(sorted_vals.iloc[0])
            second_score = float(sorted_vals.iloc[1]) if len(sorted_vals) > 1 else np.nan

            module_rows.append({
                "resolution": res,
                "cluster_key": key,
                "cluster": str(cl),
                "top_marker_module": top_module,
                "second_marker_module": second_module,
                "top_module_score": top_score,
                "second_module_score": second_score,
                "top_minus_second_module_score": float(top_score - second_score) if pd.notna(second_score) else np.nan,
            })

    marker_module_df = pd.DataFrame(module_rows)
    write_tsv_replace(marker_module_df, MARKER_MODULE_TSV)
    display(marker_module_df.head())

In [ ]:
# =========================
# 8) BUILD ADVISOR TABLE FOR RESOLUTION SELECTION
# =========================

# This scoring is only an advisor. It should not be treated as an automatic final truth.
# It balances cluster granularity, small-cluster burden, metadata bias, silhouette, and marker module separation.

advisor = sweep_df.merge(
    composition_summary_df,
    on=["resolution", "cluster_key", "n_clusters"],
    how="left"
)

# Add cluster silhouette
if "metric" in silhouette_df.columns:
    cluster_sil = silhouette_df[
        silhouette_df["metric"] == "cluster_silhouette_on_X_scVI_subsample"
    ][["resolution", "cluster_key", "value"]].rename(columns={"value": "cluster_silhouette_subsample"})
    advisor = advisor.merge(cluster_sil, on=["resolution", "cluster_key"], how="left")
else:
    advisor["cluster_silhouette_subsample"] = np.nan

# Add marker module summary
if "top_minus_second_module_score" in marker_module_df.columns:
    marker_summary = marker_module_df.groupby(["resolution", "cluster_key"], observed=True).agg(
        median_top_minus_second_module_score=("top_minus_second_module_score", "median"),
        mean_top_minus_second_module_score=("top_minus_second_module_score", "mean"),
        fraction_clusters_marker_separation_gt_0_10=("top_minus_second_module_score", lambda x: float(np.mean(np.asarray(x) > 0.10))),
        fraction_clusters_marker_separation_gt_0_20=("top_minus_second_module_score", lambda x: float(np.mean(np.asarray(x) > 0.20))),
    ).reset_index()
    advisor = advisor.merge(marker_summary, on=["resolution", "cluster_key"], how="left")
else:
    advisor["median_top_minus_second_module_score"] = np.nan
    advisor["mean_top_minus_second_module_score"] = np.nan
    advisor["fraction_clusters_marker_separation_gt_0_10"] = np.nan
    advisor["fraction_clusters_marker_separation_gt_0_20"] = np.nan

# -------------------------
# Scoring components
# -------------------------

def cluster_count_score(n):
    # Broad preference: enough granularity to separate GBM lineages/states, not too fragmented.
    # This is NOT targeting a fixed number like 24.
    if 15 <= n <= 35:
        return 1.0
    if 10 <= n < 15:
        return 0.75
    if 35 < n <= 45:
        return 0.75
    if 8 <= n < 10:
        return 0.45
    if 45 < n <= 60:
        return 0.45
    return 0.20

advisor["score_cluster_count_range"] = advisor["n_clusters"].apply(cluster_count_score)

# Penalize many cells in tiny clusters.
advisor["score_small_cluster_burden"] = 1.0 - np.minimum(1.0, advisor["frac_cells_in_clusters_lt_500"] * 10.0)

# Penalize very strong dataset dominance. Not all dataset structure is technical, so weight is moderate.
advisor["score_dataset_mixing"] = 1.0 - np.clip(
    (advisor["mean_top_dataset_fraction_weighted"] - 0.55) / 0.45,
    0,
    1,
)

# Penalize extreme patient/sample dominance. Some patient-specific tumor states can be real, so this is moderate.
advisor["score_patient_not_overdominant"] = 1.0 - np.clip(
    (advisor["mean_top_patient_fraction_weighted"] - 0.35) / 0.65,
    0,
    1,
)

# Prefer clear marker-module separation when available.
if "median_top_minus_second_module_score" in advisor.columns:
    max_marker_sep = np.nanmax(advisor["median_top_minus_second_module_score"].values)
    min_marker_sep = np.nanmin(advisor["median_top_minus_second_module_score"].values)
    denom = max(max_marker_sep - min_marker_sep, 1e-9)
    advisor["score_marker_module_separation"] = (
        advisor["median_top_minus_second_module_score"] - min_marker_sep
    ) / denom
else:
    advisor["score_marker_module_separation"] = np.nan

advisor["score_marker_module_separation"] = advisor["score_marker_module_separation"].fillna(0.5)

# Cluster silhouette: normalize across tested resolutions.
if "cluster_silhouette_subsample" in advisor.columns and advisor["cluster_silhouette_subsample"].notna().any():
    smin = advisor["cluster_silhouette_subsample"].min()
    smax = advisor["cluster_silhouette_subsample"].max()
    denom = max(smax - smin, 1e-9)
    advisor["score_cluster_silhouette_relative"] = (advisor["cluster_silhouette_subsample"] - smin) / denom
else:
    advisor["score_cluster_silhouette_relative"] = 0.5

# Final advisor score with transparent weights.
advisor["advisor_score"] = (
    0.20 * advisor["score_cluster_count_range"] +
    0.20 * advisor["score_small_cluster_burden"] +
    0.15 * advisor["score_dataset_mixing"] +
    0.10 * advisor["score_patient_not_overdominant"] +
    0.25 * advisor["score_marker_module_separation"] +
    0.10 * advisor["score_cluster_silhouette_relative"]
)

# Add human-readable flags.
flags = []
for _, r in advisor.iterrows():
    fs = []
    if r["n_clusters"] < 10:
        fs.append("very_coarse")
    if r["n_clusters"] > 45:
        fs.append("possibly_overclustered")
    if r["frac_cells_in_clusters_lt_500"] > 0.01:
        fs.append("many_cells_in_tiny_clusters")
    if r.get("mean_top_dataset_fraction_weighted", 0) > 0.75:
        fs.append("dataset_dominance_risk")
    if r.get("mean_top_patient_fraction_weighted", 0) > 0.55:
        fs.append("patient_dominance_risk")
    if r.get("median_top_minus_second_module_score", np.nan) < 0.05:
        fs.append("weak_marker_module_separation")
    flags.append(";".join(fs) if fs else "no_major_flag")

advisor["advisor_flags"] = flags

advisor = advisor.sort_values("advisor_score", ascending=False).reset_index(drop=True)
write_tsv_replace(advisor, ADVISOR_TSV)

display(advisor[[
    "resolution", "cluster_key", "n_clusters",
    "min_cluster_size", "frac_cells_in_clusters_lt_500",
    "mean_top_dataset_fraction_weighted",
    "mean_top_patient_fraction_weighted",
    "median_top_minus_second_module_score",
    "cluster_silhouette_subsample",
    "advisor_score",
    "advisor_flags",
]].head(16))

recommendation = advisor.head(5).copy()
recommendation["note"] = (
    "Top candidate resolutions by advisor_score. Final choice should also inspect UMAP, marker dotplots, composition heatmaps, and biological interpretability."
)
write_tsv_replace(recommendation, RECOMMENDATION_TSV)
display(recommendation)

In [ ]:
# =========================
# 9) DETAILED FIGURES FOR TOP CANDIDATE RESOLUTIONS
# =========================

top_candidate_keys = list(dict.fromkeys(advisor.head(N_TOP_CANDIDATE_FIGURES)["cluster_key"].tolist()))

# Always include common reference resolutions if available, but do not force them as final.
for ref_res in [0.30, 0.40, 0.50]:
    ref_key = f"leiden_scvi_HVG8000_r{str(ref_res).replace('.', '_')}"
    if ref_key in adata.obs.columns and ref_key not in top_candidate_keys:
        top_candidate_keys.append(ref_key)

print("Detailed figure cluster keys:")
print(top_candidate_keys)

# UMAP detailed for candidates
for key in top_candidate_keys:
    fig_path = FIGURES_DIR / f"clustering_QC_HVG8000_UMAP_{key}_lowmem.png"
    with plt.rc_context({"figure.figsize": (7, 6)}):
        sc.pl.umap(
            adata,
            color=key,
            legend_loc="on data",
            legend_fontsize=6,
            show=False,
        )
        plt.savefig(fig_path, dpi=220, bbox_inches="tight")
        plt.close()
    print("[SAVED]", fig_path)

# Composition heatmaps for candidates
for key in top_candidate_keys:
    comp = composition_df[composition_df["cluster_key"] == key].copy()
    if comp.empty:
        continue

    # Dataset composition
    tab_dataset = pd.crosstab(adata.obs[key].astype(str), adata.obs["core_dataset"].astype(str), normalize="index")
    tab_condition = pd.crosstab(adata.obs[key].astype(str), adata.obs["condition"].astype(str), normalize="index")

    for tab, label in [(tab_dataset, "dataset"), (tab_condition, "condition")]:
        # Sort cluster index numerically when possible
        try:
            tab = tab.loc[sorted(tab.index, key=lambda x: int(x) if str(x).isdigit() else str(x))]
        except Exception:
            pass

        fig_h = max(4, 0.25 * tab.shape[0])
        fig_w = max(6, 0.7 * tab.shape[1] + 4)
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        im = ax.imshow(tab.values, aspect="auto", vmin=0, vmax=1)
        ax.set_yticks(np.arange(tab.shape[0]))
        ax.set_yticklabels(tab.index, fontsize=7)
        ax.set_xticks(np.arange(tab.shape[1]))
        ax.set_xticklabels(tab.columns, rotation=45, ha="right", fontsize=8)
        ax.set_title(f"{label} composition per cluster — {key}")
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label("Fraction within cluster")
        plt.tight_layout()
        fig_path = FIGURES_DIR / f"clustering_QC_HVG8000_heatmap_{label}_composition_{key}_lowmem.png"
        plt.savefig(fig_path, dpi=220, bbox_inches="tight")
        plt.close()
        print("[SAVED]", fig_path)

# Marker dotplots and module heatmaps
if adata_marker is not None:
    present_marker_panels_for_dotplot = {
        module: [g for g in genes if g in adata_marker.var_names]
        for module, genes in marker_panels.items()
    }
    present_marker_panels_for_dotplot = {
        module: genes for module, genes in present_marker_panels_for_dotplot.items()
        if len(genes) > 0
    }

    for key in top_candidate_keys:
        if key not in adata_marker.obs.columns:
            continue

        # Dotplot
        try:
            dot_path = FIGURES_DIR / f"clustering_QC_HVG8000_marker_dotplot_{key}_lowmem.png"
            dp = sc.pl.dotplot(
                adata_marker,
                var_names=present_marker_panels_for_dotplot,
                groupby=key,
                standard_scale="var",
                dendrogram=False,
                show=False,
                return_fig=True,
            )
            dp.savefig(dot_path)
            plt.close("all")
            print("[SAVED]", dot_path)
        except Exception as e:
            print(f"[WARNING] Dotplot failed for {key}:", repr(e))

        # Marker module score heatmap
        sub = marker_module_df[marker_module_df["cluster_key"] == key].copy()
        if sub.empty:
            continue

        # Build matrix: clusters x marker modules from adata_marker obs score_cols
        score_cols = [c for c in adata_marker.obs.columns if c.startswith("score_")]
        module_mean = adata_marker.obs.groupby(key, observed=True)[score_cols].mean()
        module_mean.columns = [c.replace("score_", "") for c in module_mean.columns]

        try:
            module_mean = module_mean.loc[sorted(module_mean.index, key=lambda x: int(x) if str(x).isdigit() else str(x))]
        except Exception:
            pass

        # z-score per module for visualization
        mat = module_mean.copy()
        mat = (mat - mat.mean(axis=0)) / (mat.std(axis=0) + 1e-9)

        fig_h = max(5, 0.25 * mat.shape[0])
        fig_w = max(8, 0.45 * mat.shape[1])
        fig, ax = plt.subplots(figsize=(fig_w, fig_h))
        im = ax.imshow(mat.values, aspect="auto", vmin=-2, vmax=2)
        ax.set_yticks(np.arange(mat.shape[0]))
        ax.set_yticklabels(mat.index, fontsize=7)
        ax.set_xticks(np.arange(mat.shape[1]))
        ax.set_xticklabels(mat.columns, rotation=60, ha="right", fontsize=8)
        ax.set_title(f"Marker module score heatmap — {key}")
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label("Z-scored mean module score")
        plt.tight_layout()
        fig_path = FIGURES_DIR / f"clustering_QC_HVG8000_marker_module_heatmap_{key}_lowmem.png"
        plt.savefig(fig_path, dpi=220, bbox_inches="tight")
        plt.close()
        print("[SAVED]", fig_path)

else:
    print("Skipping marker dotplots because adata_marker is unavailable.")

In [ ]:
# =========================
# 10) SAVE H5AD WITH RESOLUTION SWEEP CLUSTER COLUMNS
# =========================

# This h5ad is useful for the next notebook because it contains all tested Leiden cluster columns.
# It may be large because it contains the HVG8000 matrix. If writing fails due to memory/disk,
# all essential cluster assignments are already saved in TSVs above.

try:
    write_h5ad_replace(adata, OUT_H5AD)
except Exception as e:
    print("[WARNING] Could not save full h5ad with sweep columns:", repr(e))
    print("Saving obs-only cluster assignment table instead.")
    obs_cols = ["core_dataset", "condition", "patient_id", "sample_id"] + resolution_keys
    obs_cluster_tsv = METADATA_DIR / "clustering_resolution_obs_cluster_assignments_HVG8000_lowmem.tsv"
    obs_out = adata.obs[obs_cols].copy()
    obs_out.insert(0, "cell_id", adata.obs_names)
    write_tsv_replace(obs_out, obs_cluster_tsv)

# Output manifest
manifest_rows = [
    {"name": "input_scvi_h5ad", "path": str(SCVI_H5AD), "exists": SCVI_H5AD.exists()},
    {"name": "fullgene_h5ad", "path": str(FULLGENE_H5AD), "exists": FULLGENE_H5AD.exists()},
    {"name": "output_resolution_qc_h5ad", "path": str(OUT_H5AD), "exists": OUT_H5AD.exists()},
    {"name": "input_qc_tsv", "path": str(INPUT_QC_TSV), "exists": INPUT_QC_TSV.exists()},
    {"name": "sweep_summary_tsv", "path": str(SWEEP_SUMMARY_TSV), "exists": SWEEP_SUMMARY_TSV.exists()},
    {"name": "cluster_sizes_tsv", "path": str(CLUSTER_SIZES_TSV), "exists": CLUSTER_SIZES_TSV.exists()},
    {"name": "composition_tsv", "path": str(COMPOSITION_TSV), "exists": COMPOSITION_TSV.exists()},
    {"name": "composition_summary_tsv", "path": str(COMPOSITION_SUMMARY_TSV), "exists": COMPOSITION_SUMMARY_TSV.exists()},
    {"name": "silhouette_tsv", "path": str(SILHOUETTE_TSV), "exists": SILHOUETTE_TSV.exists()},
    {"name": "marker_module_tsv", "path": str(MARKER_MODULE_TSV), "exists": MARKER_MODULE_TSV.exists()},
    {"name": "advisor_tsv", "path": str(ADVISOR_TSV), "exists": ADVISOR_TSV.exists()},
    {"name": "recommendation_tsv", "path": str(RECOMMENDATION_TSV), "exists": RECOMMENDATION_TSV.exists()},
]

manifest = pd.DataFrame(manifest_rows)
write_tsv_replace(manifest, MANIFEST_TSV)
display(manifest)

print("\nDONE. Review advisor table + UMAPs + composition heatmaps + marker dotplots before choosing the final resolution.")